# 1. Introduction

1. This case_study optimizes the quantities of **Blinky** units shipped from **2 assembly plants** to **2 distribution centers (DCs)**, and from those DCs to **3 demand regions**, with the objective of **minimizing total cost**.
2. The problem structure and input parameters come from the **MITx Supply Chain MicroMasters** (*Supply Chain Analytics / Optimization* module).
3. All given parameters are summarized in the next section.

# 2. What is given?

1. **Monthly regional demand** (in units):

| **Demand** | Region 1 | Region 2 | Region 3 |
| :--- | :---: | :---: | :---: |
| **Units / month** | 2500 | 4350 | 3296 |

2. **Distances from assembly plants to DCs** (in miles):

| **Distance (Miles)** | DC1 | DC2 |
| :--- | :---: | :---: |
| **Assembly Plant 1** | 190 | 15 |
| **Assembly Plant 2** | 150 | 36 |

3. **Inbound transportation rate** (plant to DC): **$0.04 / unit / mile**.
   
4. **Monthly plant capacities**: **13,000** units (Plant 1) and **3,500** units (Plant 2).

5. **Distances from DCs to demand regions** (in miles):

| **Distance (Miles)** | Region 1 | Region 2 | Region 3 |
| :--- | :---: | :---: | :---: |
| **DC1** | 15 | 59 | 79 |
| **DC2** | 135 | 45 | 129 |

6. **Outbound transportation rate** (DC to region): **$0.08 / unit / mile**.

7. **Monthly fixed operating costs** for DCs: **\$11,500 (DC1)** and **\$15,500 (DC2)**.

# 3. Structure

## 3.1. Import

In [1]:
import pandas as pd
import numpy as np
np.set_printoptions(suppress=True, precision=0)
from scipy.optimize import Bounds, LinearConstraint, milp
from IPython.display import display

## 3.2. Objective Function

### 3.2.1. Structure

1. The model is formulated as a Mixed-Integer Linear Program (MILP) solved via `scipy.optimize.milp`.

2. The 12-element decision vector `x` is ordered as follows:

```text
x = [
    x[0]:  Plant 1 -> DC1
    x[1]:  Plant 2 -> DC1
    x[2]:  Plant 1 -> DC2
    x[3]:  Plant 2 -> DC2
    x[4]:  DC1 -> Region 1
    x[5]:  DC1 -> Region 2
    x[6]:  DC1 -> Region 3
    x[7]:  DC2 -> Region 1
    x[8]:  DC2 -> Region 2
    x[9]:  DC2 -> Region 3
    x[10]: DC1 open/used    (binary: 0 or 1)
    x[11]: DC2 open/used    (binary: 0 or 1)
]
```

### 3.2.2. Function

In [2]:
plant_dc = np.array([190, 150, 15, 36, 0, 0, 0, 0, 0, 0, 0, 0])
dc_region = np.array([0, 0, 0, 0, 15, 59, 79, 135, 45, 129, 0, 0])
fixed_cost = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11500, 15500])
inbound_fee = 0.04
outbound_fee = 0.08

c = plant_dc * inbound_fee + dc_region * outbound_fee + fixed_cost

## 3.3. Bounds

In [3]:
# X matrix lower and upper bounds:
lb = np.zeros(12)
ub = np.concatenate([np.full(10, np.inf), np.ones(2)])
bounds = Bounds(lb=lb, ub=ub)

## 3.4. Constraints

In [4]:
# Regional Demand Constraints and Constraint Bounds
A1 = np.concatenate([np.zeros((3, 4)), np.eye(3), np.eye(3), np.zeros((3, 2))], axis=1)
lb1 = ub1 = [2500, 4350, 3296]

# Plant Capacity Constraints and Constraint Bounds 
A2 = np.array([[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]])
lb2 = np.zeros(2)
ub2 = np.array([13000, 3500])

# Conservation of Flow Constraints and Constraint Bounds
A3 = np.array([[1, 1, 0, 0, -1, -1, -1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 0, -1, -1, -1, 0, 0]])
lb3 = ub3 = np.zeros(2)

# DCs Utilization Binary Constraints and Constraint Bounds
A4 = np.array([[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, -10146, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, -10146]])
lb4 = np.full(2, -np.inf)
ub4 = np.zeros(2)

# Constraints Matrix and Bounds
A = np.vstack([A1, A2, A3, A4])
lb = np.concatenate([lb1, lb2, lb3, lb4], axis=0)
ub = np.concatenate([ub1, ub2, ub3, ub4], axis=0)

constraints = LinearConstraint(A, lb=lb, ub=ub)

## 3.5. Integrality

In [5]:
# This will be a MILP problem
# This will also render binary variables (last 2 items in X) as binary
integrality = np.ones(12)

# 4. Solution

In [6]:
results = milp(c=c, bounds=bounds, constraints=constraints, integrality=integrality)

In [7]:
if results.success:
	print("Quantities sent from plants to the DCs:")
	display(pd.DataFrame(results.x[:4].reshape((2, 2), order="F"), index=["plant1", "plant2"], columns=["DC1", "DC2"]))
	print("\n", "Quantities sent from DCs to the Regions:")
	display(pd.DataFrame(results.x[4:10].reshape((2, 3), order="C"), index=["DC1", "DC2"], columns=["Region1", "Region2", "Region3"]))
	print("\n", "Binary values indicating whether a DC is utilized:")
	display(pd.DataFrame(results.x[10:].reshape((1, 2)), index=["Binary"], columns=["DC1", "DC2"]))
	print("\n", "Minimized Transportation Cost: ", results.fun)
	print("\n", "Matrix A:")
	print(A)
	print("\n", "Array X:")
	print(results.x)
	print("\n", "Constraints - left hand side")
	print(A@results.x)
else:
	print(results.message)

Quantities sent from plants to the DCs:


,DC1,DC2
plant1,0.0,10146.0
plant2,-0.0,0.0



 Quantities sent from DCs to the Regions:


,Region1,Region2,Region3
DC1,0.0,0.0,0.0
DC2,2500.0,4350.0,3296.0



 Binary values indicating whether a DC is utilized:


,DC1,DC2
Binary,-0.0,1.0



 Minimized Transportation Cost:  98262.32

 Matrix A:
[[     0.      0.      0.      0.      1.      0.      0.      1.      0.
       0.      0.      0.]
 [     0.      0.      0.      0.      0.      1.      0.      0.      1.
       0.      0.      0.]
 [     0.      0.      0.      0.      0.      0.      1.      0.      0.
       1.      0.      0.]
 [     1.      0.      1.      0.      0.      0.      0.      0.      0.
       0.      0.      0.]
 [     0.      1.      0.      1.      0.      0.      0.      0.      0.
       0.      0.      0.]
 [     1.      1.      0.      0.     -1.     -1.     -1.      0.      0.
       0.      0.      0.]
 [     0.      0.      1.      1.      0.      0.      0.     -1.     -1.
      -1.      0.      0.]
 [     1.      1.      0.      0.      0.      0.      0.      0.      0.
       0. -10146.      0.]
 [     0.      0.      1.      1.      0.      0.      0.      0.      0.
       0.      0. -10146.]]

 Array X:
[    0.    -0. 10146.   

# 5. Key Takeaways

1. Optimized (minimized) transportation cost is obtained by utilizing plant1 and DC2 only.
2. This obliges all the regional demands be provided by DC2 only.
3. Optimized transportation cost happens to be 98262.32 dollars.